# Generative Faces with Convolutional VAE

**Project Overview:**
This notebook implements a Convolutional Variational Autoencoder trained on the CelebA dataset. Unlike a standard Autoencoder which learns a deterministic compression, a VAE learns a probability distribution over the latent space.

**Engineering Goals:**
1.  **Convolutional Architecture:** Use Strided Convolutions for downsampling and Transposed Convolutions for upsampling, since it preserves spatial hierarchies better than Linear layers.
2.  **Variational Inference:** Optimize the Evidence Lower Bound, balancing Reconstruction Loss against KL-Divergence.
3.  **Latent Space Analysis:** Demonstrate the smoothness of the learned manifold via latent space interpolation.

**Structure:**
*   `src.model`: Contains the `ConvVAE` architecture.
*   `src.loss`: Implements the ELBO loss (Reconstruction + KLD).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

sys.path.append('../src')

from dataset import CelebADataset
from model import ConvVAE
from loss import VAELoss
from utils import seed_everything, count_parameters

CONFIG = {
    "DATA_DIR": "../data/celeba",
    "WEIGHTS_DIR": "../weights",
    "LATENT_DIM": 128,
    "BATCH_SIZE": 64,
    "LR": 0.0005,
    "EPOCHS": 20,
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "SEED": 42
}

seed_everything(CONFIG['SEED'])
os.makedirs(CONFIG['WEIGHTS_DIR'], exist_ok=True)

print(f"Running on: {CONFIG['DEVICE']}")

## 1. Data Preparation

We load the CelebA dataset. The `CelebADataset` class in `src/dataset.py` handles:
1.  **Center Cropping:** Focusing on the face removing background noise.
2.  **Resizing:** Downscaling to 64x64 for efficient training.

In [ ]:
train_dataset = CelebADataset(
    root_dir=CONFIG['DATA_DIR'],
    split='train',
    image_size=64
)

val_dataset = CelebADataset(
    root_dir=CONFIG['DATA_DIR'],
    split='val',
    image_size=64
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['BATCH_SIZE'],
    shuffle=False,
    num_workers=4
)

print(f"Training Images: {len(train_dataset)}")
print(f"Validation Images: {len(val_dataset)}")

In [ ]:
def show_batch(dataloader):
    """Visualizes a batch of training data"""
    images, _ = next(iter(dataloader))
    plt.figure(figsize=(12, 6))
    plt.axis("off")
    plt.title("Training Images (Cropped & Resized)")
    plt.imshow(np.transpose(make_grid(images[:32], padding=2, normalize=True).cpu(), (1, 2, 0)))
    plt.show()

show_batch(train_loader)

## 2. Model Architecture: ConvVAE

We'll initialize the Convolutional VAE.
*   **Encoder:** 4x Strided Conv2d Layers -> Flatten -> Linear -> Gaussian Params ($\mu, \sigma$).
*   **Decoder:** Linear -> Unflatten -> 4x ConvTranspose2d Layers.

In [ ]:
model = ConvVAE(latent_dim=CONFIG['LATENT_DIM']).to(CONFIG['DEVICE'])

print(f"Model: Convolutional VAE (Latent Dim: {CONFIG['LATENT_DIM']})")
print(f"Trainable Parameters: {count_parameters(model):,}")

dummy = torch.randn(2, 3, 64, 64).to(CONFIG['DEVICE'])
recon, mu, logvar = model(dummy)
print(f"Input: {dummy.shape} -> Reconstruction: {recon.shape}")
print(f"Latent Mu: {mu.shape}")

## 3. Training Loop

We'll train the model to minimize the ELBO.
$$ \mathcal{L} = \mathcal{L}_{recon} + \beta \cdot D_{KL}(q(z|x) || p(z)) $$

*   $\mathcal{L}_{recon}$: Ensures the output looks like the input (BCE Loss).
*   $D_{KL}$: Ensures the latent space is normally distributed (KL Divergence).

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=CONFIG['LR'])
criterion = VAELoss(kld_weight=0.00025)

def run_epoch(loader, is_train=True):
    model.train() if is_train else model.eval()

    total_loss = 0
    recon_loss_accum = 0
    kld_loss_accum = 0

    pbar = tqdm(loader, desc="Train" if is_train else "Val", leave=False)

    for images, _ in pbar:
        images = images.to(CONFIG['DEVICE'])

        with torch.set_grad_enabled(is_train):
            recon_images, mu, logvar = model(images)
            loss, recon, kld = criterion(recon_images, images, mu, logvar)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        recon_loss_accum += recon.item()
        kld_loss_accum += kld.item()

        pbar.set_postfix({"Loss": loss.item()})

    avg_loss = total_loss / len(loader.dataset)
    avg_recon = recon_loss_accum / len(loader.dataset)
    avg_kld = kld_loss_accum / len(loader.dataset)

    return avg_loss, avg_recon, avg_kld

In [ ]:
TRAIN_MODEL = True

history = {"train_loss": [], "val_loss": [], "kld": []}

if TRAIN_MODEL:
    print("Starting Training...")
    for epoch in range(CONFIG['EPOCHS']):
        t_loss, t_rec, t_kld = run_epoch(train_loader, is_train=True)
        v_loss, v_rec, v_kld = run_epoch(val_loader, is_train=False)

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['kld'].append(t_kld)

        print(f"Epoch {epoch+1}/{CONFIG['EPOCHS']} | "
              f"Train Loss: {t_loss:.4f} (Rec: {t_rec:.2f}, KLD: {t_kld:.2f}) | "
              f"Val Loss: {v_loss:.4f}")

    torch.save(model.state_dict(), f"{CONFIG['WEIGHTS_DIR']}/vae_celeba.pth")
    print("Model Saved.")
else:
    if os.path.exists(f"{CONFIG['WEIGHTS_DIR']}/vae_celeba.pth"):
        model.load_state_dict(torch.load(f"{CONFIG['WEIGHTS_DIR']}/vae_celeba.pth"))
        print("Weights Loaded.")
    else:
        print("No weights found.")

In [ ]:
if TRAIN_MODEL:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Total Loss')
    plt.plot(history['val_loss'], label='Val Total Loss')
    plt.legend()
    plt.title("Total Loss (ELBO)")

    plt.subplot(1, 2, 2)
    plt.plot(history['kld'], label='KL Divergence', color='orange')
    plt.legend()
    plt.title("KL Divergence Term")
    plt.show()

## 4. Evaluation: Reconstruction Quality

First, we check if the model can reconstruct seen images. A good VAE should produce slightly blurry but structurally accurate reconstructions.

In [ ]:
model.eval()

def visualize_reconstruction(n=8):
    images, _ = next(iter(val_loader))
    images = images[:n].to(CONFIG['DEVICE'])

    with torch.no_grad():
        recon_images, _, _ = model(images)

    comparison = torch.cat([images, recon_images])

    plt.figure(figsize=(n*2, 4))
    plt.axis("off")
    plt.title("Top: Original | Bottom: Reconstruction")
    plt.imshow(np.transpose(make_grid(comparison.cpu(), nrow=n, padding=2).numpy(), (1, 2, 0)))
    plt.show()

visualize_reconstruction()

## 5. Evaluation: Generation

The true test of a VAE is sampling from the prior distribution $p(z) = \mathcal{N}(0, I)$.
If the KL-Divergence was optimized correctly, random noise passed through the decoder should result in plausible faces.

In [ ]:
def generate_faces(n=16):
    model.eval()
    z = torch.randn(n, CONFIG['LATENT_DIM']).to(CONFIG['DEVICE'])

    with torch.no_grad():
        generated_images = model.decoder_input(z)
        generated_images = generated_images.view(-1, 256, 4, 4)
        generated_images = model.decoder(generated_images)

    plt.figure(figsize=(8, 8))
    plt.axis("off")
    plt.title("Generated Faces (from Random Noise)")
    plt.imshow(np.transpose(make_grid(generated_images.cpu(), nrow=4, padding=2).numpy(), (1, 2, 0)))
    plt.show()

generate_faces()

## 6. Evaluation: Latent Space Interpolation

This is the most visually compelling demonstration. We take two real images, encode them into latent vectors $z_1$ and $z_2$, and then decode vectors along the linear path between them.

$$ z_{new} = (1 - \alpha) \cdot z_1 + \alpha \cdot z_2 $$

Smooth transitions prove that the model has learned a continuous manifold of facial features.

In [ ]:
def interpolate_points(p1, p2, n_steps=10):
    ratios = np.linspace(0, 1, num=n_steps)
    vectors = []
    for ratio in ratios:
        v = (1.0 - ratio) * p1 + ratio * p2
        vectors.append(v)
    return torch.stack(vectors)

def latent_walk(idx1=0, idx2=1):
    model.eval()

    dataset = val_loader.dataset
    img1, _ = dataset[idx1]
    img2, _ = dataset[idx2]

    with torch.no_grad():
        _, mu1, _ = model(img1.unsqueeze(0).to(CONFIG['DEVICE']))
        _, mu2, _ = model(img2.unsqueeze(0).to(CONFIG['DEVICE']))

    z_steps = interpolate_points(mu1, mu2, n_steps=10)
    with torch.no_grad():
        decoded_steps = model.decoder_input(z_steps)
        decoded_steps = decoded_steps.view(-1, 256, 4, 4)
        out_images = model.decoder(decoded_steps)

    plt.figure(figsize=(20, 4))
    plt.axis("off")
    plt.title("Latent Space Interpolation")
    plt.imshow(np.transpose(make_grid(out_images.cpu(), nrow=10, padding=2).numpy(), (1, 2, 0)))
    plt.show()

latent_walk(idx1=np.random.randint(0, 100), idx2=np.random.randint(0, 100))